# Honest evaluation of the fare model

This notebook answers one question: **how good is the model on flights it has never seen?**

It does four things:
1. Compares three ways of testing the model (random rows, new flights, later dates).
2. Adds 95% confidence intervals per class and per airline.
3. Checks where the model is systematically too high or too low.
4. Builds a prediction interval for each fare (a "likely range", not just one number).

Run all cells in order. Set the settings in the next cell first.

In [16]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

warnings.filterwarnings("ignore")

# ------------------------------ SETTINGS ------------------------------
DATA_FILES = ["Cleaned_dataset.csv", "data/Cleaned_dataset.csv"]
DATE_COL = "Date_of_journey"          # departure date column; if missing, Days_left is used instead
FLIGHT_KEY_COLS = ["Airline", "Flight_code", "Source", "Destination", "Departure", "Arrival"]
N_SPLITS = 5                          # folds for the new-flight test
TEST_SIZE = 0.2                       # share of data used for testing
RANDOM_STATE = 42
MAX_ITER = 300                        # lower (e.g. 100) to run faster
N_BOOT = 1000                         # bootstrap repeats
ALPHA = 0.2                           # 0.2 -> 80% prediction interval
MODEL_DIR = "model"
USE_MLFLOW = True                  # set True after: pip install mlflow
# ----------------------------------------------------------------------

def show(d):
    if "display" in globals():
        display(d)
    else:
        print(d.to_string())

## 1. Load and clean (same steps as the app) and print row counts

In [4]:
path = next((p for p in DATA_FILES if os.path.exists(p)), None)
assert path is not None, "Cleaned_dataset.csv not found. Put it next to this notebook."

df = pd.read_csv(path)
print(f"rows in file             : {len(df):,}")
df.columns = df.columns.str.strip()

for col in df.select_dtypes(exclude="number").columns:
    df[col] = df[col].astype(str).str.strip()

for col in ["Fare", "Duration_in_hours", "Days_left"]:
    if not pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].astype(str).str.replace(",", "", regex=False)
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["Fare", "Duration_in_hours", "Days_left"])
print(f"after removing bad values: {len(df):,}")
df = df.drop_duplicates().reset_index(drop=True)
print(f"after removing duplicates: {len(df):,}")

df["Stops"] = df["Total_stops"].str.extract(r"(\d+)")[0].astype(float).fillna(0).astype(int)
df = df[df["Fare"] > 0].reset_index(drop=True)
print(f"rows used                : {len(df):,}")
print(df["Class"].value_counts().to_string())

rows in file             : 452,088
after removing bad values: 452,088
after removing duplicates: 445,366
rows used                : 445,366
Class
Economy            248323
Business           126535
Premium Economy     70364
First                 144


## 2. Define "a flight"

Many rows are the *same flight on different dates*. If one date is in training and another in testing,
the model is partly recognising the flight, not predicting a new one. Each flight gets an id so we can keep
all its rows on the same side of the split.

In [5]:
if "Flight_code" in df.columns:
    key_cols = [c for c in FLIGHT_KEY_COLS if c in df.columns]
else:
    key_cols = ["Airline", "Source", "Destination", "Departure", "Arrival", "Stops"]
    print("Flight_code not found: a flight is defined by airline, route, times and stops instead.")

df["flight_id"] = df.groupby(key_cols, dropna=False).ngroup()
print("Flight defined by:", key_cols)
print(f"unique flights: {df['flight_id'].nunique():,} | average rows per flight: {len(df) / df['flight_id'].nunique():.1f}")

Flight defined by: ['Airline', 'Flight_code', 'Source', 'Destination', 'Departure', 'Arrival']
unique flights: 6,344 | average rows per flight: 70.2


## 3. Model and metrics

In [6]:
CAT_FEATURES = ["Airline", "Class", "Source", "Destination", "Departure", "Arrival", "Journey_day"]
NUM_FEATURES = ["Stops", "Duration_in_hours", "Days_left"]
FEATURES = CAT_FEATURES + NUM_FEATURES


def build_model():
    prep = ColumnTransformer([
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan), CAT_FEATURES),
        ("num", "passthrough", NUM_FEATURES),
    ])
    booster = HistGradientBoostingRegressor(
        max_iter=MAX_ITER, learning_rate=0.2, max_leaf_nodes=127, min_samples_leaf=20,
        l2_regularization=0.1,
        categorical_features=[True] * len(CAT_FEATURES) + [False] * len(NUM_FEATURES),
        random_state=42,
    )
    return Pipeline([
        ("prep", prep),
        ("model", TransformedTargetRegressor(regressor=booster, func=np.log1p, inverse_func=np.expm1)),
    ])


def metrics(actual, pred):
    ape = np.abs(pred - actual) / actual
    return {
        "R2": r2_score(actual, pred),
        "MAE": float(np.mean(np.abs(pred - actual))),
        "MAPE %": float(ape.mean() * 100),
        "Within 10 %": float((ape <= 0.10).mean() * 100),
        "Within 20 %": float((ape <= 0.20).mean() * 100),
    }


def same_flight_baseline(tr, te):
    # simple lookup: average fare of the same flight and class in the training rows
    train = df.iloc[tr]
    means = train.groupby(["flight_id", "Class"])["Fare"].mean().rename("m").reset_index()
    class_mean = train.groupby("Class")["Fare"].mean()
    t = df.iloc[te][["flight_id", "Class"]].merge(means, on=["flight_id", "Class"], how="left")
    t["m"] = t["m"].fillna(t["Class"].map(class_mean)).fillna(train["Fare"].mean())
    return t["m"].to_numpy()


X = df[FEATURES]
y = df["Fare"].to_numpy()
groups = df["flight_id"].to_numpy()

## 4. Three ways to test the model

| Test | What it asks | Expectation |
|---|---|---|
| Random split | Can the model price a row when the same flight is also in training? | Most optimistic |
| New flights (GroupKFold) | Can it price a flight it has never seen? | **Main number to report** |
| Later dates | Trained on earlier departure dates, tested on later ones | Hardest |

The date test is also strict because all prices come from one day, so "later dates" means "more days before
departure". Trees cannot extrapolate beyond the days-left range they saw, so treat this one as a stress test.

In [7]:
results = []
idx = np.arange(len(df))

# ---- A) random split ----
tr, te = train_test_split(idx, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=df["Class"])
pred = build_model().fit(X.iloc[tr], y[tr]).predict(X.iloc[te])
results.append({
    "Protocol": "Random split", "Test rows": len(te),
    "Seen-flight rows (%)": np.isin(groups[te], np.unique(groups[tr])).mean() * 100,
    **metrics(y[te], pred),
    "Same-flight lookup R2": r2_score(y[te], same_flight_baseline(tr, te)),
})
print("random split done")

# ---- B) GroupKFold by flight (also gives out-of-fold predictions used later) ----
oof = np.full(len(df), np.nan)
oof_base = np.full(len(df), np.nan)
for k, (tr, te) in enumerate(GroupKFold(n_splits=N_SPLITS).split(X, y, groups), start=1):
    oof[te] = build_model().fit(X.iloc[tr], y[tr]).predict(X.iloc[te])
    oof_base[te] = same_flight_baseline(tr, te)
    print(f"fold {k}/{N_SPLITS} done")
results.append({
    "Protocol": "New flights (GroupKFold)", "Test rows": len(df),
    "Seen-flight rows (%)": 0.0,
    **metrics(y, oof),
    "Same-flight lookup R2": r2_score(y, oof_base),
})

# ---- C) date split ----
dates = pd.to_datetime(df[DATE_COL], dayfirst=True, errors="coerce") if DATE_COL in df.columns else None
if dates is not None and dates.notna().mean() > 0.5:
    order_key, how = dates.fillna(dates.max()), f"departure date ({DATE_COL})"
else:
    order_key, how = df["Days_left"], "Days_left (a larger value means a later departure)"
cutoff = order_key.sort_values().iloc[int(len(df) * (1 - TEST_SIZE))]
tr = idx[(order_key <= cutoff).to_numpy()]
te = idx[(order_key > cutoff).to_numpy()]
print(f"date split uses {how}; train rows {len(tr):,}, test rows {len(te):,}")
pred = build_model().fit(X.iloc[tr], y[tr]).predict(X.iloc[te])
results.append({
    "Protocol": "Later dates", "Test rows": len(te),
    "Seen-flight rows (%)": np.isin(groups[te], np.unique(groups[tr])).mean() * 100,
    **metrics(y[te], pred),
    "Same-flight lookup R2": r2_score(y[te], same_flight_baseline(tr, te)),
})

summary = pd.DataFrame(results).round(
    {"R2": 3, "MAE": 0, "MAPE %": 1, "Within 10 %": 1, "Within 20 %": 1,
     "Seen-flight rows (%)": 1, "Same-flight lookup R2": 3}
)
show(summary)

random split done
fold 1/5 done
fold 2/5 done
fold 3/5 done
fold 4/5 done
fold 5/5 done
date split uses departure date (Date_of_journey); train rows 356,564, test rows 88,802
                   Protocol  Test rows  Seen-flight rows (%)     R2     MAE  MAPE %  Within 10 %  Within 20 %  Same-flight lookup R2
0              Random split      89074                  99.9  0.953  2386.0    10.8         60.5         84.2                  0.922
1  New flights (GroupKFold)     445366                   0.0  0.909  3540.0    16.8         44.4         69.5                  0.799
2               Later dates      88802                  99.9  0.964  1972.0    10.2         63.5         86.2                  0.925


### How to read this table
- **Seen-flight rows (%)** is the share of test rows whose flight also appears in training. It should be high for the random split (your mentor saw about 96%) and 0 for the new-flight test.
- **Same-flight lookup R2** is a simple "average of the same flight" guess. If the model beats it by a lot on new flights, it is learning real price patterns.
- Report the **New flights** row as the headline and keep the random-split row next to it for comparison.

## 5. Confidence intervals per class and per airline

A single MAE hides how uncertain it is, especially for small groups. This resamples **whole flights**
(not single rows), because rows of one flight are not independent. It uses the out-of-fold predictions from the
new-flight test above.

In [8]:
def cluster_bootstrap(kind, labels, actual, pred, grp, n_boot=N_BOOT, seed=0):
    labels = np.asarray(labels)
    seg_codes, seg_names = pd.factorize(labels)
    S = len(seg_names)
    g_codes, g_uni = pd.factorize(grp)
    G = len(g_uni)
    ae = np.abs(pred - actual)
    ape = ae / actual
    hit = (ape <= 0.10).astype(float)
    rng = np.random.default_rng(seed)
    boots = {k: np.empty((n_boot, S)) for k in ("MAE", "MAPE %", "Within 10 %")}
    for b in range(n_boot):
        w = np.bincount(rng.integers(0, G, G), minlength=G)[g_codes].astype(float)
        den = np.bincount(seg_codes, weights=w, minlength=S)
        with np.errstate(invalid="ignore", divide="ignore"):
            boots["MAE"][b] = np.bincount(seg_codes, weights=w * ae, minlength=S) / den
            boots["MAPE %"][b] = np.bincount(seg_codes, weights=w * ape, minlength=S) / den * 100
            boots["Within 10 %"][b] = np.bincount(seg_codes, weights=w * hit, minlength=S) / den * 100
    rows = []
    for s in range(S):
        m = seg_codes == s
        flights = int(pd.Series(grp[m]).nunique())
        row = {"Group": kind, "Segment": seg_names[s], "Rows": int(m.sum()), "Flights": flights}
        point = {"MAE": ae[m].mean(), "MAPE %": ape[m].mean() * 100, "Within 10 %": hit[m].mean() * 100}
        for k in ("MAE", "MAPE %", "Within 10 %"):
            lo, hi = np.nanpercentile(boots[k][:, s], [2.5, 97.5])
            row[k] = f"{point[k]:,.1f}"
            row[k + " (95% CI)"] = f"{lo:,.1f} to {hi:,.1f}"
        row["Note"] = "very few flights: treat as rough" if flights < 50 else ""
        rows.append(row)
    return pd.DataFrame(rows)


ci = pd.concat([
    cluster_bootstrap("All", np.repeat("All flights", len(df)), y, oof, groups),
    cluster_bootstrap("Class", df["Class"].to_numpy(), y, oof, groups),
    cluster_bootstrap("Airline", df["Airline"].to_numpy(), y, oof, groups),
], ignore_index=True)
show(ci)

      Group          Segment    Rows  Flights       MAE          MAE (95% CI) MAPE % MAPE % (95% CI) Within 10 % Within 10 % (95% CI)                              Note
0       All      All flights  445366     6344   3,539.8    3,459.5 to 3,621.5   16.8    16.5 to 17.2        44.4         43.6 to 45.2                                  
1     Class          Economy  248323     6090   1,813.9    1,775.9 to 1,849.9   18.1    17.7 to 18.5        41.8         40.8 to 42.6                                  
2     Class  Premium Economy   70364     1488   2,821.6    2,728.1 to 2,918.4   17.0    16.3 to 17.7        45.1         43.7 to 46.6                                  
3     Class         Business  126535     2510   7,306.4    7,098.2 to 7,534.4   14.2    13.8 to 14.7        49.4         48.0 to 50.7                                  
4     Class            First     144        4  20,866.2  13,859.7 to 42,071.5   31.8    27.0 to 44.3         0.0           0.0 to 0.0  very few flights: treat a

## 6. Is the model systematically too high or too low?

Average error can look fine while one group is consistently over- or under-priced. This shows the **average signed
error in %** (positive = the model prices too high) on flights it had not seen.

In [9]:
d = pd.DataFrame({
    "Class": df["Class"],
    "Airline": df["Airline"],
    "Route": df["Source"] + " to " + df["Destination"],
    "Days left": pd.cut(df["Days_left"], [-1, 3, 7, 14, 21, 30, 10_000],
                        labels=["0-3", "4-7", "8-14", "15-21", "22-30", "31+"]),
    "err": (oof - y) / y * 100,
})

for col in ["Class", "Airline", "Days left"]:
    t = d.groupby(col, observed=True)["err"].agg(["mean", "size"]).round(1)
    t.columns = ["Average error (%)", "Rows"]
    print(f"--- by {col} ---")
    show(t)

routes = d.groupby("Route")["err"].agg(["mean", "size"])
routes = routes[routes["size"] >= 300].round(1)
routes.columns = ["Average error (%)", "Rows"]
print("--- routes the model prices too high ---")
show(routes.nlargest(5, "Average error (%)"))
print("--- routes the model prices too low ---")
show(routes.nsmallest(5, "Average error (%)"))

--- by Class ---
                 Average error (%)    Rows
Class                                     
Business                       1.3  126535
Economy                        2.6  248323
First                        -31.8     144
Premium Economy                1.7   70364
--- by Airline ---
             Average error (%)    Rows
Airline                               
Air India                  1.1  118405
AirAsia                    2.7   15764
AkasaAir                   1.3    2230
AllianceAir                5.1     393
GO FIRST                   2.8   10786
Indigo                     3.1   65659
SpiceJet                   3.0    5863
StarAir                  -15.5      62
Vistara                    2.1  226204
--- by Days left ---
           Average error (%)    Rows
Days left                           
0-3                      3.1   24824
4-7                      3.1   34135
8-14                     2.3   62751
15-21                    2.4   63942
22-30                    1.8   817

## 7. Prediction intervals (conformal prediction)

Instead of one number, give a range that should contain the real fare about 80% of the time.

How it works, in plain terms:
1. Train the model on one set of flights.
2. On a separate set of flights (the calibration set), measure how far off the model usually is.
3. Use that typical miss to put a range around every new prediction.

First class has very few flights, so it is **pooled with Business** when measuring the typical miss.
The model and the numbers are saved together, because the range is only valid for the model it was measured on.

In [10]:
rest, te = next(GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE).split(X, y, groups))
tr_rel, cal_rel = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE).split(
    X.iloc[rest], y[rest], groups[rest]))
tr, cal = rest[tr_rel], rest[cal_rel]
print(f"train rows {len(tr):,} | calibration rows {len(cal):,} | test rows {len(te):,} (flights do not overlap)")

model_v2 = build_model().fit(X.iloc[tr], y[tr])

seg = df["Class"].replace({"First": "Business"})        # pool First with Business
miss = lambda actual, pred: np.abs(np.log1p(actual) - np.log1p(pred))

s_cal = miss(y[cal], model_v2.predict(X.iloc[cal]))


def conformal_q(scores):
    n = len(scores)
    level = min(1.0, np.ceil((n + 1) * (1 - ALPHA)) / n)
    return float(np.quantile(scores, level, method="higher"))


q = {name: conformal_q(g.to_numpy()) for name, g in pd.Series(s_cal).groupby(seg.iloc[cal].to_numpy())}
q_all = conformal_q(s_cal)
print({k: round(v, 3) for k, v in q.items()}, "| fallback", round(q_all, 3))


def interval(pred, seg_names):
    qv = np.array([q.get(s, q_all) for s in seg_names])
    lo = np.maximum(np.expm1(np.log1p(pred) - qv), 0)
    hi = np.expm1(np.log1p(pred) + qv)
    return lo, hi


p_te = model_v2.predict(X.iloc[te])
lo, hi = interval(p_te, seg.iloc[te].to_numpy())
inside = (y[te] >= lo) & (y[te] <= hi)

cov = pd.DataFrame({
    "Class": df["Class"].iloc[te].to_numpy(),
    "Covered": inside,
    "Width (% of prediction)": (hi - lo) / p_te * 100,
    "Flight": groups[te],
})
cov_table = cov.groupby("Class").agg(
    **{"Test rows": ("Covered", "size"), "Test flights": ("Flight", "nunique"),
       "Actual fare inside range (%)": ("Covered", lambda s: s.mean() * 100),
       "Typical range width (% of prediction)": ("Width (% of prediction)", "median")}
).round(1)
cov_table.loc["All"] = [len(te), int(cov["Flight"].nunique()), round(inside.mean() * 100, 1),
                        round(cov["Width (% of prediction)"].median(), 1)]
print(f"Target: {int((1 - ALPHA) * 100)}% of real fares inside the range")
show(cov_table)

train rows 257,561 | calibration rows 96,862 | test rows 90,943 (flights do not overlap)
{'Business': 0.223, 'Economy': 0.281, 'Premium Economy': 0.258} | fallback 0.258
Target: 80% of real fares inside the range
                 Test rows  Test flights  Actual fare inside range (%)  Typical range width (% of prediction)
Class                                                                                                        
Business           25652.0         510.0                          76.7                                   44.9
Economy            50898.0        1220.0                          76.3                                   57.0
First                 77.0           2.0                          64.9                                   44.9
Premium Economy    14316.0         292.0                          73.3                                   52.2
All                90943.0        1269.0                          75.9                                   57.0


Read the coverage column against the target. Small gaps are normal. For First class the test set is tiny, so its
coverage number can swing a lot and should not be quoted on its own.

In [11]:
os.makedirs(MODEL_DIR, exist_ok=True)
joblib.dump(model_v2, f"{MODEL_DIR}/flight_fare_model_v2.joblib")
with open(f"{MODEL_DIR}/conformal_q.json", "w") as f:
    json.dump({
        "interval_level": 1 - ALPHA,
        "note": "range = expm1(log1p(prediction) -/+ q); q chosen per class group",
        "q_by_class": q,
        "q_fallback": q_all,
        "pooled_into": {"First": "Business"},
    }, f, indent=2)
print(f"saved {MODEL_DIR}/flight_fare_model_v2.joblib and {MODEL_DIR}/conformal_q.json")
print("In Colab, download them from the Files panel before the session ends.")

saved model/flight_fare_model_v2.joblib and model/conformal_q.json
In Colab, download them from the Files panel before the session ends.


## 8. Optional: log this run with MLflow

In [18]:
if USE_MLFLOW:
    import mlflow
    mlflow.set_experiment("flight-fare")
    with mlflow.start_run(run_name="honest-evaluation"):
        mlflow.log_params({"n_splits": N_SPLITS, "test_size": TEST_SIZE, "max_iter": MAX_ITER,
                           "alpha": ALPHA, "rows": len(df), "flights": int(df["flight_id"].nunique())})
        short = {"Random split": "random", "New flights (GroupKFold)": "new_flights", "Later dates": "later_dates"}
        for _, r in summary.iterrows():
            tag = short[r["Protocol"]]
            mlflow.log_metrics({f"{tag}_r2": float(r["R2"]), f"{tag}_mae": float(r["MAE"]),
                                f"{tag}_within10": float(r["Within 10 %"])})
        mlflow.log_metric("interval_coverage", float(inside.mean() * 100))
        mlflow.log_artifact(f"{MODEL_DIR}/conformal_q.json")
        mlflow.sklearn.log_model(
    model_v2,
    name="model",
    skops_trusted_types=[
        "functools.partial",
        "sklearn.compose._column_transformer._RemainderColsList",
        "sklearn.ensemble._hist_gradient_boosting.predictor.TreePredictor",
        "sklearn.utils.validation.check_array",
    ],
)
    print("logged to MLflow")
else:
    print("MLflow logging skipped (USE_MLFLOW = False).")

logged to MLflow


## 9. Sentence for the dashboard and report

In [13]:
g = summary[summary["Protocol"].str.startswith("New")].iloc[0]
r = summary[summary["Protocol"] == "Random split"].iloc[0]
print(f"On flights the model has never seen: R2 {g['R2']:.2f}, average error about Rs {g['MAE']:,.0f}, "
      f"and {g['Within 10 %']:.0f}% of predictions are within 10% of the listed fare.")
print(f"On a random split (same flights also in training) the same model scores R2 {r['R2']:.2f} "
      f"and {r['Within 10 %']:.0f}% within 10%.")

On flights the model has never seen: R2 0.91, average error about Rs 3,540, and 44% of predictions are within 10% of the listed fare.
On a random split (same flights also in training) the same model scores R2 0.95 and 60% within 10%.


In [19]:
# --- Does Duration_in_hours include layover time? ---
print("Duration (hours) by number of stops")
show(df.groupby("Stops")["Duration_in_hours"].agg(["median", "mean", "size"]).round(1))

# Same routes, non-stop vs 1 stop: a big gap means layovers are included
d = df[df["Stops"].isin([0, 1])].copy()
d["Route"] = d["Source"] + " to " + d["Destination"]
by_route = d.groupby(["Route", "Stops"])["Duration_in_hours"].median().unstack("Stops").dropna()
by_route.columns = ["Non-stop (h)", "1 stop (h)"]
by_route["Extra hours with 1 stop"] = by_route["1 stop (h)"] - by_route["Non-stop (h)"]
print("\nSame routes: median duration, non-stop vs 1 stop")
show(by_route.round(1).sort_values("Extra hours with 1 stop", ascending=False).head(10))

# Route-mix check: average flight length per airline
print("\nAverage flight duration by airline")
show(df.groupby("Airline")["Duration_in_hours"].agg(["mean", "size"]).round(1).sort_values("mean"))

Duration (hours) by number of stops
       median  mean    size
Stops                      
0         2.2   2.1   51755
1        12.2  13.2  366527
2        14.8  17.3   27084

Same routes: median duration, non-stop vs 1 stop
                        Non-stop (h)  1 stop (h)  Extra hours with 1 stop
Route                                                                    
Bangalore to Chennai             1.1        13.8                     12.7
Bangalore to Hyderabad           1.3        13.8                     12.4
Mumbai to Ahmedabad              1.2        13.2                     12.0
Ahmedabad to Mumbai              1.3        12.9                     11.6
Hyderabad to Chennai             1.4        12.9                     11.5
Chennai to Bangalore             1.0        12.2                     11.2
Hyderabad to Bangalore           1.3        12.5                     11.2
Ahmedabad to Hyderabad           1.8        12.9                     11.1
Kolkata to Hyderabad             2